# Chess Coach — Agent d'aide à l'apprentissage des ouvertures

**Auteur :** Benoit Girard
**Environnement :** Python 3.12, paquet `chess_coach` (uv)

Ce notebook déroule, étape par étape, le raisonnement derrière l'agent. Il
manipule directement les briques du paquet `chess_coach` afin de rendre lisible la
démarche :

0. Configuration et imports
1. Représenter une position d'échecs (FEN)
2. La théorie des ouvertures (livre local)
3. Préparer la base de connaissances Wikichess (chunking)
4. Générer des embeddings
5. L'agent LangGraph : orchestrer les outils
6. Récapitulatif

> Les briques « pures » s'exécutent telles quelles. Les outils nécessitant des
> services externes (Stockfish, Milvus, MongoDB) tournent dans la stack Docker ;
> on en explique ici la logique.

## 0. Configuration et imports

In [ ]:
from pathlib import Path

from chess_coach.config import get_settings

# Le notebook vit dans notebooks/ ; le code Python vit dans backend/.
# On calcule une fois pour toutes la racine du dépôt, puis on s'y réfère.
RACINE = Path.cwd()
if not (RACINE / "docker-compose.yml").exists():
    RACINE = RACINE.parent

DOSSIER_WIKICHESS = RACINE / "backend" / "data" / "wikichess"
DOSSIER_OPENINGS = RACINE / "backend" / "data" / "openings"

settings = get_settings()
print("Racine du dépôt     :", RACINE)
print("Modèle d'embedding  :", settings.embedding_model)
print("Dimension           :", settings.embedding_dim)
print("Collection Milvus   :", settings.milvus_collection)

## 1. Représenter une position d'échecs (FEN)

Avant de raisonner, l'agent doit *comprendre* la position. À l'image de la façon
dont une position est transmise à un LLM (cf. Kaggle Game Arena), on en extrait
une description structurée : trait, coups légaux, échiquier.

In [ ]:
from chess_coach.services.chess_position import STARTING_FEN, describe_position

info = describe_position(STARTING_FEN)
print("Trait aux       :", info.side_to_move)
print(
    "Coups légaux     :",
    len(info.legal_moves_san),
    "->",
    info.legal_moves_san[:6],
    "...",
)
print(info.board_ascii)

**Observation.** Depuis la position initiale, l'agent dispose de 20 coups légaux
et d'un échiquier exploitable. Cette description sera passée aux outils et, le
cas échéant, au LLM de synthèse.

## 2. La théorie des ouvertures (livre local)

Pour un coup *théorique*, on interroge un livre d'ouvertures construit à partir
des grandes lignes. (En production, l'API Lichess enrichit ces coups avec les
statistiques de parties dès qu'un jeton est fourni.)

In [ ]:
from chess_coach.services.opening_book import OpeningBook

result = OpeningBook.lookup(STARTING_FEN)
print("Ouverture :", result.opening_name, f"({result.opening_eco})")
print("Coups théoriques :", [m.san for m in result.moves])

**Observation.** Le livre reconnaît la position et propose les premiers coups
maîtres (e4, d4, c4, Cf3). Les transpositions sont gérées car les positions sont
indexées par leur EPD.

## 3. Préparer la base de connaissances (chunking)

La base a deux dossiers : les articles téléchargés depuis Wikichess,
le répertoire collaboratif de FICGS, et des fiches rédigées en français qui les
complètent. La qualité du RAG dépend du découpage : on charge les articles,
puis on les segmente en passages avec recouvrement.


In [ ]:
from chess_coach.rag.preprocess import build_chunks, load_articles

articles = load_articles(DOSSIER_WIKICHESS) + load_articles(DOSSIER_OPENINGS)
chunks = build_chunks(articles)
print(f"{len(articles)} articles -> {len(chunks)} chunks")
print("Exemple de chunk :")
print(chunks[0].text[:200], "...")

**Observation.** Chaque chunk garde le nom de l'ouverture et son dossier
d'origine. C'est ce qui permet, dans l'interface, de citer la provenance d'un
passage : un article Wikichess ou une fiche française.


## 4. Générer des embeddings

On transforme un texte en vecteur dense normalisé. Le modèle est multilingue
(les articles sont en français).

In [ ]:
from chess_coach.services.embeddings import EmbeddingService

embedder = EmbeddingService(settings)
vectors = embedder.embed(["La défense sicilienne", "Le gambit dame"])
print("Nombre de vecteurs :", len(vectors))
print("Dimension          :", len(vectors[0]))

**Observation.** Chaque phrase devient un vecteur de 384 dimensions. Indexés
dans Milvus (recherche par produit scalaire ≡ cosinus), ils permettent la
recherche sémantique exposée par `GET /api/v1/vector-search`.

## 5. L'agent LangGraph : orchestrer les outils

L'agent est un graphe de décision. Pour une position :

1. **identify** — décrit la position (FEN valide ?) ;
2. **theory** — cherche les coups théoriques ;
3. **branche** — position connue → *théorie*, sinon → *moteur Stockfish* ;
4. **context** — enrichit avec le RAG Wikichess ;
5. **videos** — propose des vidéos YouTube ;
6. **synthesize** — rédige une recommandation en français ;
7. **persist** — enregistre l'interaction dans MongoDB.

On illustre la dernière étape (la synthèse) à partir d'un état fabriqué, sans
service externe.

In [ ]:
from chess_coach.agent.synthesize import build_template_recommendation

etat = {
    "fen": "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3",
    "in_theory": True,
    "opening_name": "Italian Game",
    "opening_eco": "C50",
    "position": {
        "side_to_move": "black",
        "legal_moves_san": ["Nf6", "Bc5", "d6"],
        "board_ascii": "(diagramme)",
    },
    "theory_moves": [
        {"san": "Nf6", "total": 103572421},
        {"san": "Bc5", "total": 101156348},
    ],
    "reference_games": [
        {"white": "Carlsen", "black": "Caruana", "result": "1-0", "year": 2019},
    ],
    "passages": [
        {"opening": "Giuoco Piano", "text": "Le fou en c4 vise la case f7."},
    ],
    "videos": [{"title": "Tutoriel"}],
}

print(build_template_recommendation(etat))

**Observation.** À partir des faits collectés par les nœuds, le gabarit
produit déjà une réponse lisible. C'est le filet de sécurité de l'agent :
il fonctionne sans aucune clé d'API.


## 6. Ce que l'on donne au modèle de langage

Le dernier nœud confie la rédaction à un modèle. On ne lui envoie pas le FEN
tout seul : comme dans les parties Kaggle Game Arena, on lui décrit la
position (échiquier, trait, coups légaux), puis on ajoute les faits réunis
par les nœuds précédents.


In [ ]:
from chess_coach.agent.synthesize import build_llm_prompt

print(build_llm_prompt(etat))

**Observation.** Le modèle ne décide rien, il met en forme. Les coups
viennent de Lichess, l'évaluation de Stockfish, le contexte de Milvus. Si
l'appel échoue, le gabarit reprend la main et l'agent répond quand même.


## 7. Récapitulatif

- La position est **comprise** : le FEN devient une description structurée.
- La **théorie** et les **parties de référence** viennent de l'Opening
  Explorer de Lichess ; le livre local prend le relais en cas de panne.
- Hors théorie, **Stockfish** évalue la position.
- Le **RAG Milvus** sur Wikichess apporte le contexte.
- L'**API YouTube** propose des vidéos.
- Un **modèle de langage** rédige la recommandation à partir de ces faits.

Le tout est orchestré par **LangGraph**, exposé par **FastAPI**, persisté dans
**MongoDB** et présenté par l'interface **Angular**.
